# Model Evaluation Multi-Metric Dashboard

## Objective
Compute a comprehensive validation matrix against the held-out 
validation set. A single metric (MAP@7) is insufficient for a 
banking recommendation system with 24 product classes and severe 
class imbalance.

## Metrics computed
1. **MAP@7** — primary ranking metric (Kaggle competition standard)
2. **Classification report** — per-class precision, recall, F1
3. **Global AUC-ROC** — multi-class discrimination (One-vs-Rest, macro)
4. **Catalog coverage** — diversity: what % of 24 products are recommended
5. **Deployment gate** — pass/fail decision before Flask serving

## Why each metric matters
- MAP@7 alone hides per-product blind spots in the model
- AUC-ROC exposes whether the model discriminates or just rank-orders
- Coverage catches filter bubbles caused by class imbalance
- Per-class F1 shows which products are being systematically ignored

## Environment and Load Artifacts

Load the trained XGBoost model and validation data produced by 
Notebooks 05 and 06. All evaluation runs against the held-out 
validation set only never the training set.

In [15]:
# import necessary libraries
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
import warnings
import os
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

# Metrics for evaluation
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Load trained model
model = xgb.Booster()
model.load_model('../artifacts/xgboost_model.json')
print("Model loaded successfully.")

# Load validation DMatrix and true labels
dval       = xgb.DMatrix('../artifacts/dval.buffer')
y_val      = np.load('../artifacts/y_val.npy')
y_train    = np.load('../artifacts/y_train.npy')

# Load product map for readable class names
import joblib
PRODUCT_COLS = joblib.load('../artifacts/feature_cols.pkl')

# Define the 24 product names for report labels
PRODUCT_NAMES = [
    "ind_ahor_fin_ult1", "ind_aval_fin_ult1", "ind_cco_fin_ult1",
    "ind_cder_fin_ult1", "ind_cno_fin_ult1",  "ind_ctju_fin_ult1",
    "ind_ctma_fin_ult1", "ind_ctop_fin_ult1", "ind_ctpp_fin_ult1",
    "ind_deco_fin_ult1", "ind_deme_fin_ult1", "ind_dela_fin_ult1",
    "ind_ecue_fin_ult1", "ind_fond_fin_ult1", "ind_hip_fin_ult1",
    "ind_plan_fin_ult1", "ind_pres_fin_ult1", "ind_reca_fin_ult1",
    "ind_tjcr_fin_ult1", "ind_valo_fin_ult1", "ind_viv_fin_ult1",
    "ind_nomina_ult1",   "ind_nom_pens_ult1", "ind_recibo_ult1"
]

print(f"Validation samples : {len(y_val):,}")
print(f"Unique true classes : {len(np.unique(y_val))}")

Model loaded successfully.
Validation samples : 6,774
Unique true classes : 18


## Generate Raw Probability Predictions

`model.predict(dval)` returns a matrix of shape (N, 24) where each 
row is the softmax probability distribution across all 24 products 
for one customer. Each row sums to 1.0.

Hard predictions (argmax) pick the single most probable product 
used for classification report and coverage metrics.
Soft predictions (the full matrix) are used for AUC-ROC and MAP@7.

In [16]:
# Raw softmax probability matrix shape (N, 24)
raw_preds = model.predict(dval)

# Hard predictions single most likely product per customer
hard_preds = np.argmax(raw_preds, axis=1)

print(f"Prediction matrix shape : {raw_preds.shape}")
print(f"Hard predictions shape  : {hard_preds.shape}")
print(f"Probability row sum check (first 3 rows): "
      f"{raw_preds[:3].sum(axis=1).round(4)}")
# Each row should sum to ~1.0 — confirms softmax is working

Prediction matrix shape : (6774, 24)
Hard predictions shape  : (6774,)
Probability row sum check (first 3 rows): [1. 1. 1.]


## MAP@7: Primary Ranking Metric

Mean Average Precision at K=7. For each customer we take the 7 
highest-probability products and check how many of the products 
they actually added appear in that list, with higher positions 
weighted more heavily.

This is the official Kaggle competition metric for this dataset.
Target threshold: MAP@7 ≥ 0.03 (competition baseline).
A well-tuned model reaches 0.028–0.032 range.

In [17]:
# MAP@7 evaluation metric implementation
# MAP@7 is the mean of the average precision scores at K=7 across all customers.
def mapk(actual, predicted, k=7):
    """
    Compute Mean Average Precision at K.
    
    Args:
        actual   : list of lists — true product indices per customer
        predicted: list of lists — ranked predicted product indices per customer
        k        : cutoff rank
    Returns:
        float: MAP@K score
    """
    # Average Precision at K for a single customer
    # This function computes the average precision for one customer based on their true products and the predicted ranking.
    def apk(a, p, k):
        if not a:
            return 0.0
        p = p[:k]
        score, hits = 0.0, 0
        for i, pred in enumerate(p):
            if pred in a and pred not in p[:i]:
                hits += 1
                score += hits / (i + 1)
        return score / min(len(a), k)

    return np.mean([
        apk([a], p, k)
        for a, p in zip(actual, predicted)
    ])

# Build top-7 ranked product indices per customer
top7_preds = np.argsort(raw_preds, axis=1)[:, ::-1][:, :7].tolist()

map7_score = mapk(y_val.tolist(), top7_preds, k=7)
print(f"MAP@7 Score : {map7_score:.6f}")
print(f"Target      : ≥ 0.028 (competition baseline)")
print(f"Status      : {' PASS' if map7_score >= 0.028 else ' BELOW BASELINE review training'}")

MAP@7 Score : 0.699674
Target      : ≥ 0.028 (competition baseline)
Status      :  PASS


## Classification Report Per-Class Precision, Recall, F1

Using hard predictions (argmax) to evaluate per-product accuracy.

Key things to look for:
- Products with F1 = 0.00 are being completely ignored by the model
- Products with very high recall but low precision are being over-recommended
- Support column shows how many true examples exist per product in validation

Low F1 on high-support products is a model problem.
Low F1 on low-support products may be acceptable given the 9852x imbalance.

In [18]:
# Determine which product indices actually appear in validation targets
present_classes = sorted(np.unique(y_val).tolist())
present_names   = [PRODUCT_NAMES[i] for i in present_classes]

# Generate classification report for hard predictions
print("Classification Report Per-Product Performance")
print("=" * 65)
print(classification_report(
    y_val,
    hard_preds,
    labels=present_classes,
    target_names=present_names,
    zero_division=0
))

Classification Report Per-Product Performance
                   precision    recall  f1-score   support

 ind_cco_fin_ult1       0.73      0.87      0.79       609
 ind_cno_fin_ult1       0.37      0.57      0.45       435
ind_ctju_fin_ult1       1.00      0.88      0.93         8
ind_ctma_fin_ult1       0.12      0.54      0.19        28
ind_ctop_fin_ult1       0.30      0.48      0.37        56
ind_ctpp_fin_ult1       0.30      0.28      0.29        32
ind_dela_fin_ult1       0.00      0.00      0.00         8
ind_ecue_fin_ult1       0.62      0.68      0.65       487
ind_fond_fin_ult1       0.00      0.00      0.00        12
 ind_hip_fin_ult1       0.00      0.00      0.00         2
ind_plan_fin_ult1       0.00      0.00      0.00         4
ind_reca_fin_ult1       0.10      0.36      0.15        45
ind_tjcr_fin_ult1       0.66      0.69      0.67       840
ind_valo_fin_ult1       0.18      0.21      0.19        43
 ind_viv_fin_ult1       0.00      0.00      0.00         3
  ind_nom

## Global AUC-ROC Multi-Class Discrimination

We normalise the filtered probability matrix so each row sums to 
1.0 before passing to roc_auc_score. This is required because 
slicing 18 columns from a 24-class softmax output breaks the 
probability constraint that roc_auc_score enforces.

Normalisation does not change the ranking it only rescales the 
probabilities to satisfy the mathematical requirement.

In [22]:
# Classes present in validation labels
present_classes = sorted(np.unique(y_val).tolist())
print(f"Classes in y_val: {len(present_classes)} → {present_classes}")

# Slice only the columns for classes present in val
raw_preds_filtered = raw_preds[:, present_classes]

# CRITICAL FIX: re-normalise rows to sum to 1.0
# Slicing 18 of 24 softmax columns breaks the probability constraint
# Dividing each row by its sum restores it without changing rankings
row_sums = raw_preds_filtered.sum(axis=1, keepdims=True)
raw_preds_normalised = raw_preds_filtered / row_sums

print(f"Row sum check after normalisation "
      f"(first 3): {raw_preds_normalised[:3].sum(axis=1).round(6)}")
# Should print [1.0, 1.0, 1.0]

# Re-index y_val to 0-based positions matching filtered columns
class_to_idx    = {cls: idx for idx, cls in enumerate(present_classes)}
y_val_reindexed = np.array([class_to_idx[v] for v in y_val])

# Compute AUC-ROC
global_auc = roc_auc_score(
    y_val_reindexed,
    raw_preds_normalised,
    multi_class='ovr',
    average='macro'
)

print(f"\nGlobal AUC-ROC (macro OvR) : {global_auc:.4f}")
print(f"Target                        : ≥ 0.70")
print(f"Status : {'PASS' if global_auc >= 0.70 else 'BELOW TARGET'}")

Classes in y_val: 18 → [2, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23]
Row sum check after normalisation (first 3): [1. 1. 1.]

Global AUC-ROC (macro OvR) : 0.8942
Target                        : ≥ 0.70
Status : PASS


## Deployment Gate Pass/Fail Summary

All three metrics use live computed values from this notebook session.
No placeholder values. Any failure blocks deployment.

In [24]:
# All three variables must be computed from cells above no placeholders
print(f"Live metric values entering gate:")
print(f"  map7_score       : {map7_score:.6f}")
print(f"  global_auc       : {global_auc:.4f}")
print(f"  catalog_coverage : {catalog_coverage:.2f}%")

# Deployment gate
gates = {
    "MAP@7 ≥ 0.028"          : map7_score      >= 0.028,
    "AUC-ROC ≥ 0.70"         : global_auc      >= 0.70,
    "Catalog Coverage ≥ 50%" : catalog_coverage >= 50.0,
}

print("\n" + "=" * 50)
print("  DEPLOYMENT GATE — METRIC SUMMARY")
print("=" * 50)

# Evaluate each gate and print results
all_pass = True
for metric, passed in gates.items():
    status   = " PASS" if passed else " FAIL"
    all_pass = all_pass and passed
    print(f"  {status}  {metric}")

print("=" * 50)
if all_pass:
    print("  ALL GATES PASSED!  model approved for deployment")
else:
    print("  ONE OR MORE GATES FAILED, do not deploy")
    print("  Review class weights, hyperparameters, or training data")
print("=" * 50)

# Log to MLflow
import mlflow
with mlflow.start_run(run_name="evaluation_notebook07"):
    mlflow.log_metric("map_at_7",         map7_score)
    mlflow.log_metric("auc_roc_macro",    global_auc)
    mlflow.log_metric("catalog_coverage", catalog_coverage)
    mlflow.log_param("deployment_ready",  str(all_pass))

print("\nAll metrics logged to MLflow.")

Live metric values entering gate:
  map7_score       : 0.699674
  global_auc       : 0.8942
  catalog_coverage : 55.40%

  DEPLOYMENT GATE — METRIC SUMMARY
   PASS  MAP@7 ≥ 0.028
   PASS  AUC-ROC ≥ 0.70
   PASS  Catalog Coverage ≥ 50%
  ALL GATES PASSED!  model approved for deployment

All metrics logged to MLflow.
